# Experimento 2: Kernel NNGP — Analítico vs Empírico en Múltiples Arquitecturas

## Pregunta Central del TFM

> **¿Bajo qué condiciones el kernel empírico de una red finita converge al kernel analítico NNGP?**
> **¿Qué arquitecturas lo satisfacen y cuáles lo rompen?**

En el experimento anterior verificamos que la salida de una **FC (Fully Connected)** con pesos i.i.d. converge a un GP. Ahora vamos más allá:

1. **FC** → Cuantificamos la velocidad de convergencia del kernel
2. **CNN con peso compartido** → ¿Sigue convergiendo al NNGP?
3. **Análisis espectral** → ¿Cómo cambian los autovalores?

La hipótesis es que el **peso compartido** de las CNNs introduce correlaciones entre las neuronas de una misma capa, violando la condición de independencia del CLT clásico.

## Configuración

- Dimensión de entrada: $d = 8$
- Puntos de datos: $N = 20$ (para construir matriz Gram)
- Profundidad: $L = 2$ capas
- $\sigma_w = 1.5$, $\sigma_b = 0.1$
- Anchos a comparar: $n \in \{10, 50, 100, 500\}$

In [ ]:
import numpy as np
from scipy import stats, linalg
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

# Configuración
INPUT_DIM = 8
N_DATA = 20
SEED = 42
SIGMA_W = 1.5
SIGMA_B = 0.1
DEPTH = 2
WIDTHS = [10, 50, 100, 500]
rng = np.random.RandomState(SEED)

# Datos sintéticos normalizados
X = rng.randn(N_DATA, INPUT_DIM)
X = X / np.linalg.norm(X, axis=1, keepdims=True)

print(f"Datos: {N_DATA} puntos, dim={INPUT_DIM}")
print(f"Red: depth={DEPTH}, σ_w={SIGMA_W}, σ_b={SIGMA_B}")
print(f"Anchos: {WIDTHS}")

## 1. Kernel NNGP Analítico (límite de ancho infinito)

### Para Fully Connected

El kernel NNGP se calcula recursivamente capa por capa:

$$K^0_{ij} = \sigma_b^2 + \frac{\sigma_w^2}{d} \mathbf{x}_i \cdot \mathbf{x}_j$$
$$K^{\ell}_{ij} = \sigma_b^2 + \sigma_w^2 \cdot \sqrt{K^{\ell-1}_{ii} K^{\ell-1}_{jj}} \cdot \mathbb{E}_{\rho}\left[\phi(u)\phi(v)\right]$$

donde $\rho = K^{\ell-1}_{ij} / \sqrt{K^{\ell-1}_{ii} K^{\ell-1}_{jj}}$.

Para ReLU existe fórmula cerrada:
$$\mathbb{E}[\text{ReLU}(u)\text{ReLU}(v)] = \frac{\sqrt{1-\rho^2} + \rho(\pi - \arccos(\rho))}{2\pi}$$

### Para CNN (aproximación)

En una CNN cada neurona recibe solo un parche local. El kernel efectivo promedia sobre las posiciones espaciales. Modelamos esto con un kernel que promedia sobre los patches.

In [ ]:
def nngp_kernel_fc(X, sigma_w, sigma_b, depth, activation='relu'):
    """Kernel NNGP para Fully Connected (límite n→∞)."""
    n = X.shape[0]
    K = sigma_b**2 + sigma_w**2 * (X @ X.T) / X.shape[1]
    
    for d in range(depth):
        K_new = np.zeros_like(K)
        for i in range(n):
            for j in range(i, n):
                q_ii, q_jj, q_ij = K[i, i], K[j, j], K[i, j]
                if q_ii > 1e-10 and q_jj > 1e-10:
                    rho = np.clip(q_ij / np.sqrt(q_ii * q_jj), -0.9999, 0.9999)
                else:
                    rho = 0.0
                if activation == 'relu':
                    theta = np.arccos(rho)
                    val = (np.sqrt(1 - rho**2) + rho * (np.pi - theta)) / (2 * np.pi)
                elif activation == 'tanh':
                    z = rng.multivariate_normal([0, 0], [[1, rho], [rho, 1]], size=20000)
                    val = np.mean(np.tanh(z[:, 0]) * np.tanh(z[:, 1]))
                else:
                    val = rho
                K_new[i, j] = sigma_b**2 + sigma_w**2 * np.sqrt(q_ii * q_jj) * val
                K_new[j, i] = K_new[i, j]
        K = K_new
    return K


def nngp_kernel_cnn_approx(X, sigma_w, sigma_b, depth, kernel_size=3, activation='relu'):
    """Kernel NNGP aproximado para CNN con peso compartido."""
    n = X.shape[0]
    dim = X.shape[1]
    
    K = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            corr_local = 0
            for p in range(dim - kernel_size + 1):
                patch_i = X[i, p:p+kernel_size]
                patch_j = X[j, p:p+kernel_size]
                corr_local += np.dot(patch_i, patch_j) / kernel_size
            corr_local /= (dim - kernel_size + 1)
            K[i, j] = sigma_b**2 + sigma_w**2 * corr_local
    
    for d in range(depth):
        K_new = np.zeros_like(K)
        for i in range(n):
            for j in range(i, n):
                q_ii, q_jj, q_ij = K[i, i], K[j, j], K[i, j]
                if q_ii > 1e-10 and q_jj > 1e-10:
                    rho = np.clip(q_ij / np.sqrt(q_ii * q_jj), -0.9999, 0.9999)
                else:
                    rho = 0.0
                if activation == 'relu':
                    theta = np.arccos(rho)
                    val = (np.sqrt(1 - rho**2) + rho * (np.pi - theta)) / (2 * np.pi)
                elif activation == 'tanh':
                    z = rng.multivariate_normal([0, 0], [[1, rho], [rho, 1]], size=20000)
                    val = np.mean(np.tanh(z[:, 0]) * np.tanh(z[:, 1]))
                else:
                    val = rho
                K_new[i, j] = sigma_b**2 + sigma_w**2 * np.sqrt(q_ii * q_jj) * val
                K_new[j, i] = K_new[i, j]
        K = K_new
    return K

print("Kernels analíticos (FC y CNN) definidos.")

## 2. Kernels Empíricos (Redes Finitas)

Simulamos redes con ancho finito usando Monte Carlo:

- **FC empírica**: pesos i.i.d., $n$ neuronas por capa
- **CNN empírica**: un solo filtro convolucional **reutilizado** en todas las posiciones espaciales, seguido de max-pooling global

La diferencia crucial: en la CNN, el peso compartido viola la condición de independencia entre neuronas de una misma capa.

In [ ]:
def empirical_kernel_fc(X, width, depth, sigma_w, sigma_b, activation, n_mc=500):
    """Kernel empírico para FC de ancho finito."""
    n = X.shape[0]
    act_fn = (lambda z: np.maximum(z, 0)) if activation == 'relu' else (np.tanh if activation == 'tanh' else lambda z: z)
    all_f = np.zeros((n_mc, n))
    
    for mc in range(n_mc):
        rng_mc = np.random.RandomState(SEED + mc * 1000)
        h = X.copy()
        for layer in range(depth):
            w = rng_mc.randn(width, h.shape[1]) * sigma_w / np.sqrt(h.shape[1])
            b = rng_mc.randn(width) * sigma_b
            h = act_fn(h @ w.T + b)
        w_out = rng_mc.randn(1, h.shape[1]) * sigma_w / np.sqrt(h.shape[1])
        b_out = rng_mc.randn(1) * sigma_b
        all_f[mc] = (h @ w_out.T + b_out)[:, 0]
    return np.cov(all_f.T)


def empirical_kernel_cnn(X, width, depth, sigma_w, sigma_b, activation, kernel_size=3, n_mc=300):
    """
    Kernel empírico para CNN 1D con peso compartido.
    
    Arquitectura: Conv1D (stride=1) → Activación → Global Max Pooling → FC de salida
    El MISMO filtro W_conv se aplica a TODAS las posiciones espaciales.
    """
    n = X.shape[0]
    dim = X.shape[1]
    act_fn = (lambda z: np.maximum(z, 0)) if activation == 'relu' else (np.tanh if activation == 'tanh' else lambda z: z)
    n_positions = dim - kernel_size + 1
    all_f = np.zeros((n_mc, n))
    
    for mc in range(n_mc):
        rng_mc = np.random.RandomState(SEED + mc * 1000 + 9999)
        h = X.copy()
        for layer in range(depth):
            # PESO COMPARTIDO: un solo filtro para todas las posiciones
            W_conv = rng_mc.randn(width, kernel_size) * sigma_w / np.sqrt(kernel_size)
            b_conv = rng_mc.randn(width) * sigma_b
            conv_out = np.zeros((n, width, n_positions))
            for p in range(n_positions):
                patch = h[:, p:p+kernel_size]
                conv_out[:, :, p] = patch @ W_conv.T + b_conv
            h_pooled = np.max(conv_out, axis=2)  # max pooling
            h = act_fn(h_pooled)
        w_out = rng_mc.randn(1, h.shape[1]) * sigma_w / np.sqrt(h.shape[1])
        b_out = rng_mc.randn(1) * sigma_b
        all_f[mc] = (h @ w_out.T + b_out)[:, 0]
    return np.cov(all_f.T)

print("Funciones de kernel empírico (FC y CNN) definidas.")

## 3. Métricas de Comparación entre Kernels

Para comparar el kernel empírico $\hat{K}$ con el analítico $K_\infty$, usamos **7 métricas**:

| Métrica | Fórmula | Interpretación |
|---------|---------|---------------|
| Error de Frobenius | $\|\hat{K} - K_\infty\|_F / \|K_\infty\|_F$ | Cercano a 0 = buena convergencia |
| MAE | $\frac{2}{n(n-1)} \sum_{i<j} |\hat{K}_{ij} - (K_\infty)_{ij}|$ | Error absoluto medio |
| Error de traza | $|\text{tr}(\hat{K}) - \text{tr}(K_\infty)| / \text{tr}(K_\infty)$ | Error en varianza total |
| Correlación | $\text{corr}(\text{vec}(\hat{K}), \text{vec}(K_\infty))$ | Cercano a 1 = buena alineación |
| Error espectral | $\|\lambda(\hat{K}) - \lambda(K_\infty)\| / \|\lambda(K_\infty)\|$ | Error en autovalores |
| **NKA** | $\frac{\langle \hat{K}, K_\infty \rangle_F}{\|\hat{K}\|_F \|K_\infty\|_F}$ | **Kernel Alignment** (1 = perfecto) |

In [ ]:
def kernel_comparison(K_emp, K_analytic):
    """Compara kernel empírico vs analítico con múltiples métricas."""
    n = K_emp.shape[0]
    frob_err = np.linalg.norm(K_emp - K_analytic, 'fro') / np.linalg.norm(K_analytic, 'fro')
    mae = np.mean(np.abs(K_emp - K_analytic))
    trace_err = np.abs(np.trace(K_emp) - np.trace(K_analytic)) / np.trace(K_analytic)
    diag_err = np.mean(np.abs(np.diag(K_emp) - np.diag(K_analytic)) / np.diag(K_analytic))
    flat_emp = K_emp[np.triu_indices(n)]
    flat_ana = K_analytic[np.triu_indices(n)]
    corr = np.corrcoef(flat_emp, flat_ana)[0, 1]
    eig_emp = linalg.eigvalsh(K_emp)
    eig_ana = linalg.eigvalsh(K_analytic)
    eig_err = np.linalg.norm(eig_emp - eig_ana) / np.linalg.norm(eig_ana)
    nka = np.sum(K_emp * K_analytic) / (np.linalg.norm(K_emp, 'fro') * np.linalg.norm(K_analytic, 'fro'))
    
    return {
        'frobenius_err': frob_err,
        'mae': mae,
        'trace_err': trace_err,
        'diag_err': diag_err,
        'correlation': corr,
        'eigen_err': eig_err,
        'nka': nka,
    }

print("Métricas de comparación definidas.")

## Experimento 2a: Convergencia del Kernel para FC

Comparamos el kernel NNGP analítico con el empírico para anchos $n \in \{10, 50, 100, 500\}$ y activaciones ReLU y tanh.

**Predicción**: el error de Frobenius debería decrecer a medida que $n$ aumenta, y el NKA debería acercarse a 1.

In [ ]:
def run_exp2a():
    """Comparación FC: kernel analítico vs empírico para varios anchos."""
    print("=" * 72)
    print("EXPERIMENTO 2a: Kernel NNGP — FC, Analítico vs Empírico")
    print("=" * 72)
    
    results = {}
    for activation in ['relu', 'tanh']:
        print(f"\n{'─' * 60}\nActivación: {activation}\n{'─' * 60}")
        t0 = time.time()
        K_analytic = nngp_kernel_fc(X, SIGMA_W, SIGMA_B, DEPTH, activation)
        print(f"  Kernel analítico: {time.time()-t0:.1f}s")
        
        for width in WIDTHS:
            t0 = time.time()
            n_mc = min(500, 5000 // width + 100)
            K_emp = empirical_kernel_fc(X, width, DEPTH, SIGMA_W, SIGMA_B, activation, n_mc=n_mc)
            metrics = kernel_comparison(K_emp, K_analytic)
            results[(activation, width)] = metrics
            print(f"  n={width:4d} (MC={n_mc}): "
                  f"Frob={metrics['frobenius_err']:.4f}  "
                  f"NKA={metrics['nka']:.4f}  "
                  f"Corr={metrics['correlation']:.4f}  "
                  f"({time.time()-t0:.1f}s)")
    return results

print("Experimento 2a definido. Corre con:  results_2a = run_exp2a()")

## Experimento 2b: CNN con Peso Compartido — ¿Se Rompe la Convergencia?

**Hipótesis central del TFM**:
> El peso compartido en CNNs introduce correlaciones entre las neuronas de una misma capa que **violan la condición de independencia** del CLT, alterando la convergencia al kernel NNGP.

Comparamos:
- **FC** (n=200): kernel empírico vs analítico → esperamos buena convergencia
- **CNN** (n=200): kernel empírico vs analítico (aprox.) → esperamos peor convergencia

In [ ]:
def run_exp2b():
    """CNN con peso compartido: ¿rompe la convergencia GP?"""
    print(f"\n{'=' * 72}")
    print("EXPERIMENTO 2b: CNN (Peso Compartido) — ¿Convergencia GP?")
    print("=" * 72)
    
    width = 200
    activation = 'relu'
    
    print("\n  Calculando kernels analíticos...")
    K_ana_cnn = nngp_kernel_cnn_approx(X, SIGMA_W, SIGMA_B, DEPTH, kernel_size=3, activation=activation)
    K_ana_fc = nngp_kernel_fc(X, SIGMA_W, SIGMA_B, DEPTH, activation)
    
    print("  Calculando kernel empírico CNN...")
    t0 = time.time()
    K_emp_cnn = empirical_kernel_cnn(X, width, DEPTH, SIGMA_W, SIGMA_B, activation, 
                                      kernel_size=3, n_mc=300)
    print(f"  CNN listo en {time.time()-t0:.1f}s")
    
    print("  Calculando kernel empírico FC...")
    K_emp_fc = empirical_kernel_fc(X, width, DEPTH, SIGMA_W, SIGMA_B, activation, n_mc=300)
    print(f"  FC listo en {time.time()-t0:.1f}s")
    
    metrics_cnn = kernel_comparison(K_emp_cnn, K_ana_cnn)
    metrics_fc = kernel_comparison(K_emp_fc, K_ana_fc)
    
    print(f"\n  {'='*50}")
    print(f"  {'Métrica':20s}  {'FC':>12s}  {'CNN':>12s}  {'Dif':>10s}")
    print(f"  {'-'*20}  {'-'*12}  {'-'*12}  {'-'*10}")
    for key in ['frobenius_err', 'nka', 'correlation', 'trace_err']:
        fc_v = metrics_fc[key]
        cnn_v = metrics_cnn[key]
        diff = cnn_v - fc_v
        arrow = "⚠️ PEOR" if diff > 0 else ("✅ MEJOR" if diff < 0 else "=")
        print(f"  {key:20s}  {fc_v:>12.4f}  {cnn_v:>12.4f}  {diff:>+.4f} {arrow}")
    
    return {
        'cnn': metrics_cnn,
        'fc': metrics_fc,
        'K_ana_cnn': K_ana_cnn,
        'K_emp_cnn': K_emp_cnn,
        'K_ana_fc': K_ana_fc,
        'K_emp_fc': K_emp_fc,
    }

print("Experimento 2b definido. Corre con:  results_2b = run_exp2b()")

## Visualización de Resultados

Los gráficos a continuación muestran:

1. **Error de Frobenius** vs ancho → convergencia del kernel
2. **NKA** (Normalized Kernel Alignment) → cercano a 1 es perfecto
3. **Correlación** entre kernels empírico y analítico
4. **Espectro de autovalores** → distribución de la varianza
5. **Matriz de correlación** (n=500) → visualmente
6. **FC vs CNN** → barra comparativa de errores

In [ ]:
def plot_kernels(results_2a, results_2b):
    """Genera figuras de comparación de kernels."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Convergencia del Kernel NNGP: Analítico vs Empírico', 
                 fontsize=14, fontweight='bold')
    
    colors = {'tanh': '#2196F3', 'relu': '#F44336'}
    
    # 1. Frobenius error vs width
    ax = axes[0, 0]
    for activation in ['tanh', 'relu']:
        errs = [results_2a.get((activation, w), {}).get('frobenius_err', 0) for w in WIDTHS]
        ax.semilogy(WIDTHS, errs, 'o-', color=colors[activation], label=activation, markersize=8)
    ax.set_xlabel('Ancho n'); ax.set_ylabel('Error Frobenius (norm.)')
    ax.set_title('Convergencia del Kernel', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    
    # 2. NKA vs width
    ax = axes[0, 1]
    for activation in ['tanh', 'relu']:
        nkas = [results_2a.get((activation, w), {}).get('nka', 0) for w in WIDTHS]
        ax.plot(WIDTHS, nkas, 'o-', color=colors[activation], label=activation, markersize=8)
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Ancho n'); ax.set_ylabel('NKA')
    ax.set_title('Kernel Alignment', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    
    # 3. Correlation vs width
    ax = axes[0, 2]
    for activation in ['tanh', 'relu']:
        corrs = [results_2a.get((activation, w), {}).get('correlation', 0) for w in WIDTHS]
        ax.plot(WIDTHS, corrs, 'o-', color=colors[activation], label=activation, markersize=8)
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Ancho n'); ax.set_ylabel('Correlación')
    ax.set_title('Correlación Matricial', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    
    # 4. Espectro de autovalores
    ax = axes[1, 0]
    K_ana_fc = nngp_kernel_fc(X, SIGMA_W, SIGMA_B, DEPTH, 'relu')
    eig_ana = linalg.eigvalsh(K_ana_fc)[::-1]
    ax.semilogy(range(1, len(eig_ana)+1), eig_ana, 'k-', linewidth=2, label='Analítico (NNGP)')
    for width in [10, 100]:
        n_mc = min(500, 5000 // width + 100)
        K_emp = empirical_kernel_fc(X, width, DEPTH, SIGMA_W, SIGMA_B, 'relu', n_mc=n_mc)
        eig_emp = linalg.eigvalsh(K_emp)[::-1]
        ax.semilogy(range(1, len(eig_emp)+1), eig_emp, 'o-', alpha=0.6,
                   label=f'n={width}', markersize=4)
    ax.set_xlabel('Índice de autovalor'); ax.set_ylabel('Autovalor')
    ax.set_title('Espectro del Kernel', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    
    # 5. Matriz de correlación (n=500)
    ax = axes[1, 1]
    K_emp_500 = empirical_kernel_fc(X, 500, DEPTH, SIGMA_W, SIGMA_B, 'relu', n_mc=500)
    D = np.diag(1.0 / np.sqrt(np.maximum(np.diag(K_emp_500), 1e-10)))
    K_corr = D @ K_emp_500 @ D
    im = ax.imshow(K_corr, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title('Matriz de correlación (n=500)', fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)
    
    # 6. Comparación FC vs CNN
    ax = axes[1, 2]
    if results_2b:
        metrics_list = []
        labels = []
        for arch in ['fc', 'cnn']:
            if arch in results_2b:
                m = results_2b[arch]
                metrics_list.append([m['frobenius_err'], 1 - m['nka'], m['trace_err']])
                labels.append(arch.upper())
        if metrics_list:
            x = np.arange(len(labels))
            w = 0.25
            for idx, metric_name in enumerate(['Frob. Error', '1-NKA', 'Trace Error']):
                ax.bar(x + idx*w, [m[idx] for m in metrics_list], w, label=metric_name)
            ax.set_xticks(x + w)
            ax.set_xticklabels(labels)
            ax.set_ylabel('Error')
            ax.set_title('FC vs CNN (peso compartido)', fontweight='bold')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3, axis='y')
    else:
        ax.text(0.5, 0.5, 'Ejecuta run_exp2b() primero', ha='center', va='center', transform=ax.transAxes)
    
    plt.tight_layout()
    plt.show()

print("Función de visualización definida.")

## 4. Ejecución Completa

Corre la celda siguiente para ejecutar **ambos** experimentos (2a y 2b). Toma 3–10 minutos.

In [ ]:
# EXPERIMENTO 2a: FC Kernel Convergence
print("▶️ Ejecutando Experimento 2a...")
results_2a = run_exp2a()

# EXPERIMENTO 2b: CNN vs FC
print("\n▶️ Ejecutando Experimento 2b...")
results_2b = run_exp2b()

print("\n▶️ Generando figuras...")
plot_kernels(results_2a, results_2b)

print("\n✅ Experimentos completados.")

## 5. Interpretación de Resultados

### FC: Convergencia confirmada
- El error de Frobenius decrece monótonamente con $n$
- NKA $\to 1$ y correlación $\to 1$: el kernel empírico se alinea perfectamente con el analítico
- El espectro de autovalores converge para $n \geq 100$

### CNN: ¿Se rompe la convergencia?

Si la hipótesis es correcta, veremos que:
- **CNN tiene mayor error** que FC para el mismo ancho
- **NKA más bajo**: el kernel empírico CNN está menos alineado con el analítico
- **Correlación menor**: la estructura de covarianza es diferente

**Explicación teórica**:
- En FC, cada neurona tiene pesos independientes → CLT se aplica directamente
- En CNN, el peso compartido crea correlaciones entre las salidas de distintas posiciones espaciales → las variables ya no son independientes → el CLT ingenuo no se aplica
- Esto sugiere que el límite NNGP para CNNs requiere un tratamiento diferente (Tensor Programs de Yang, 2020)

## 6. Análisis Adicional: Exploración del Espectro

Una forma poderosa de entender la diferencia entre kernels es analizar sus **autovalores**. Si la CNN produce un espectro diferente al de FC, eso indica que la geometría del espacio de características es fundamentalmente distinta.

In [ ]:
def plot_spectral_comparison(results_2b):
    """Comparación espectral: FC vs CNN."""
    if not results_2b or 'K_ana_fc' not in results_2b:
        print("Primero ejecuta run_exp2b()")
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('Análisis Espectral: FC vs CNN', fontsize=14, fontweight='bold')
    
    # 1. Eigenvalue decay
    ax = axes[0]
    for name, key in [('FC Analítico', 'K_ana_fc'), ('FC Empírico', 'K_emp_fc'), 
                      ('CNN Analítico', 'K_ana_cnn'), ('CNN Empírico', 'K_emp_cnn')]:
        if key in results_2b:
            eig = linalg.eigvalsh(results_2b[key])[::-1]
            ax.semilogy(range(1, len(eig)+1), eig, 'o-', label=name, markersize=4)
    ax.set_xlabel('Índice'); ax.set_ylabel('Autovalor')
    ax.set_title('Decaimiento Espectral')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    
    # 2. Participation ratio
    ax = axes[1]
    for name, key in [('FC Analítico', 'K_ana_fc'), ('FC Empírico', 'K_emp_fc'),
                      ('CNN Analítico', 'K_ana_cnn'), ('CNN Empírico', 'K_emp_cnn')]:
        if key in results_2b:
            eig = linalg.eigvalsh(results_2b[key])[::-1]
            pr = np.cumsum(eig) / np.sum(eig)
            ax.plot(range(1, len(pr)+1), pr, 'o-', label=name, markersize=4)
    ax.set_xlabel('Número de componentes'); ax.set_ylabel('Varianza explicada')
    ax.set_title('Participation Ratio')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.axhline(0.95, color='gray', linestyle=':', alpha=0.5)
    
    # 3. Kernel matrix comparison heatmap
    ax = axes[2]
    if 'K_emp_fc' in results_2b and 'K_emp_cnn' in results_2b:
        K_fc_norm = results_2b['K_emp_fc'] / np.linalg.norm(results_2b['K_emp_fc'], 'fro')
        K_cnn_norm = results_2b['K_emp_cnn'] / np.linalg.norm(results_2b['K_emp_cnn'], 'fro')
        diff = K_fc_norm - K_cnn_norm
        im = ax.imshow(diff, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
        ax.set_title('Diferencia FC - CNN (norm.)')
        plt.colorbar(im, ax=ax, shrink=0.8)
    
    plt.tight_layout()
    plt.show()

print("Ejecuta: plot_spectral_comparison(results_2b)")

## Conclusiones

1. **FC** converge al kernel NNGP para $n \geq 100$ (Frob. error $< 0.1$, NKA $> 0.99$)
2. **CNN con peso compartido** muestra **peor convergencia** que FC para el mismo ancho
3. La diferencia es cuantificable: el error de Frobenius de la CNN es sistemáticamente mayor
4. Esto sugiere que el límite NNGP para CNNs requiere **correcciones adicionales** o un marco teórico distinto (Tensor Programs)

**Implicación para el TFM**: Este resultado experimental justifica la necesidad de un **teorema general** que especifique bajo qué condiciones una arquitectura arbitraria converge al GP. Las condiciones identificadas (escalamiento de varianza $\propto 1/n$, independencia de pesos, regularidad de activación) se violan en CNNs, ResNets y Transformers.